# Análise e Preparação de Dados de Processo Químico Industrial

## Contexto
Este notebook realiza a análise exploratória, limpeza e preparação de um dataset real de um processo químico industrial. O objetivo é validar a hipótese de que modelos baseados em árvores (ex: XGBoost, Random Forest) são superiores à Regressão Linear para prever a densidade de saída do reator (`AT-1100.PV`).

## Estrutura
1. **Ingestão dos Dados**: Leitura e limpeza inicial.
2. **Análise Exploratória (EDA)**: Estatísticas, distribuições e correlações.
3. **Engenharia de Atributos**: Tratamento de flutuações e criação de métricas de "tempo fora do ideal".
4. **Preparação para ML**: Separação de features e target.
5. **Caminhos de ML**: Proposta teórica de modelagem.

---


## 1. Ingestão dos Dados

Nesta etapa, importamos as bibliotecas necessárias e carregamos o dataset `dataset.csv`.
Realizamos também a limpeza inicial dos nomes das colunas (removendo unidades e espaços) e a conversão da coluna temporal.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Configurações globais
sns.set_theme(style="whitegrid")
custom_palette = ["#3d1152", "#6a2c70", "#b83b5e", "#f08a5d", "#f9ed69"]
sns.set_palette(sns.color_palette(custom_palette))
plt.rcParams['figure.figsize'] = (14, 7)
warnings.filterwarnings('ignore')

# Função para carregar e limpar dados
def load_data(filepath):
    # Leitura com separador ';' e decimal ','
    df = pd.read_csv(filepath, sep=';', decimal=',')
    
    # Limpeza dos nomes das colunas
    clean_columns = {}
    for col in df.columns:
        # Remove ' Value <unidade>' mantendo apenas o tag
        if ' Value' in col:
            clean_name = col.split(' Value')[0]
        else:
            clean_name = col
        clean_columns[col] = clean_name.strip()
    
    df = df.rename(columns=clean_columns)
    
    # Conversão de timestamp
    if 'Timestamp' in df.columns:
        df['Timestamp'] = pd.to_datetime(df['Timestamp'])
        # Ordenar pelo tempo para garantir integridade em séries temporais
        df = df.sort_values('Timestamp').reset_index(drop=True)
    
    return df

# Carregamento
df = load_data('dataset.csv')

# Inspeção Inicial
print(f"Shape do dataset: {df.shape}")
print("Tipos de dados:")
print(df.dtypes.value_counts())
print("\nPrimeiras linhas:")
display(df.head())


## 2. Análise Exploratória de Dados (EDA)

O objetivo desta seção é entender o comportamento das variáveis, identificar a qualidade dos dados e preparar o terreno para a modelagem. Focaremos nas variáveis de processo (`.PV`) e no target `AT-1100.PV`.



In [ ]:
# Separação de variáveis
target = 'AT-1100.PV'
cols_pv = [c for c in df.columns if '.PV' in c]
cols_sp = [c for c in df.columns if '.SP' in c]

print(f"Total de variáveis PV (Process Values): {len(cols_pv)}")
print(f"Total de variáveis SP (Set Points): {len(cols_sp)}")

# Estatísticas Descritivas das variáveis de processo
# Transposta para melhor visualização
stats_desc = df[cols_pv].describe().T
print("Estatísticas Descritivas (Top 10 variância):")
display(stats_desc.sort_values('std', ascending=False).head(10))


### Distribuição do Target (`AT-1100.PV`)

Analisamos a distribuição da variável alvo para entender sua variância e normalidade. Sendo um processo industrial contínuo, espera-se que opere em torno de um ponto de operação, mas buscamos prever as flutuações.



In [ ]:
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
sns.histplot(df[target], kde=True, color="#3d1152", bins=30)
plt.title(f'Distribuição de {target}')
plt.xlabel('Densidade de Saída (kg/m3)')

plt.subplot(1, 2, 2)
sns.lineplot(x=df['Timestamp'], y=df[target], color="#3d1152")
plt.title(f'Série Temporal de {target}')
plt.xlabel('Tempo')
plt.ylabel('Valor')

plt.tight_layout()
plt.show()


### Análise de Correlação

Verificamos a correlação entre as variáveis de processo (`.PV`) para identificar multicolinearidade e relação com o target. Variáveis com correlação muito alta entre si (redundantes) podem ser candidatas a remoção.


In [ ]:
# Matriz de correlação apenas para PVs
corr_matrix = df[cols_pv].corr()

# Plot do Heatmap
plt.figure(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='magma', vmax=1, vmin=-1, center=0,
            square=True, linewidths=.5, cbar_kws={"shrink": .5})
plt.title('Matriz de Correlação - Variáveis de Processo (.PV)')
plt.show()

# Correlações com o Target
target_corr = corr_matrix[target].sort_values(ascending=False)
print("Top 10 Correlações Positivas com Target:")
print(target_corr.head(10))
print("\nTop 10 Correlações Negativas com Target:")
print(target_corr.tail(10))


### Análise de Outliers

Utilizamos Boxplots para visualizar a dispersão das variáveis principais. Em processos industriais, "outliers" estatísticos muitas vezes representam estados transitórios reais do processo e não erros de medição, portanto **não** devem ser removidos cegamente.


In [ ]:
# Selecionando as variáveis mais correlacionadas para inspeção de outliers
top_corr_vars = target_corr.abs().sort_values(ascending=False).head(9).index.tolist()
# Remover o próprio target da lista de features para plotar
if target in top_corr_vars:
    top_corr_vars.remove(target)

plt.figure(figsize=(15, 10))
for i, col in enumerate(top_corr_vars[:6]): # Plotar top 6 features
    plt.subplot(2, 3, i+1)
    sns.boxplot(y=df[col], color="#3d1152")
    plt.title(col)
    plt.ylabel('')

plt.suptitle('Boxplots das Principais Variáveis Correlacionadas', fontsize=16)
plt.tight_layout()
plt.show()


## 3. Engenharia de Atributos e Limpeza

Nesta etapa, criamos novas features que capturam a dinâmica do processo e removemos dados redundantes.

### 3.1 Tratamento de Flutuações e Set Points
As variáveis de Set Point (`.SP`) são valores programados e fixos por longos períodos, tendo pouca variância preditiva direta. Porém, a diferença entre o Valor de Processo (`.PV`) e o Set Point (`Erro = PV - SP`) é rica em informação sobre a estabilidade do controle.

Vamos:
1. Calcular o erro para cada par PV/SP.
2. Definir uma métrica de "Tempo Fora do Ideal": Contagem de minutos, numa janela móvel de 30 minutos, em que o erro excedeu 2 desvios padrão.



In [ ]:
# Identificando pares PV e SP
pv_vars = [c for c in df.columns if '.PV' in c and c != target] # Excluir target se tiver SP
sp_vars = [c for c in df.columns if '.SP' in c]

# Dicionário para mapear PV -> SP
pv_to_sp = {}
for pv in pv_vars:
    sp_candidate = pv.replace('.PV', '.SP')
    if sp_candidate in df.columns:
        pv_to_sp[pv] = sp_candidate

print(f"Pares PV-SP identificados: {len(pv_to_sp)}")

# Engenharia de Features
df_eng = df.copy()

# Features Temporais
df_eng['Hour'] = df_eng['Timestamp'].dt.hour
df_eng['DayOfWeek'] = df_eng['Timestamp'].dt.dayofweek

# Janela para cálculo de instabilidade (ex: 30 minutos)
window_size = 30 

for pv, sp in pv_to_sp.items():
    # 1. Calcular o Erro (Flutuação)
    error_col = f"{pv}_Error"
    df_eng[error_col] = df_eng[pv] - df_eng[sp]
    
    # 2. Calcular 'Tempo Fora do Ideal'
    # Definir limiar estatístico de normalidade (ex: média +/- 2 std do erro)
    # Em produção, esses limites viriam de especificações de engenharia.
    mu, sigma = df_eng[error_col].mean(), df_eng[error_col].std()
    
    # Se desvio padrão for zero (varíavel constante), ignorar
    if sigma == 0:
        continue
        
    upper_limit = mu + 2 * sigma
    lower_limit = mu - 2 * sigma
    
    # Flag booleana: está fora do ideal?
    is_out = ((df_eng[error_col] > upper_limit) | (df_eng[error_col] < lower_limit)).astype(int)
    
    # Feature: Quantos minutos fora do ideal na última janela
    out_time_col = f"{pv}_OutTime_{window_size}m"
    df_eng[out_time_col] = is_out.rolling(window=window_size, min_periods=1).sum()

print(f"Novas features criadas. Shape atual: {df_eng.shape}")
display(df_eng[[col for col in df_eng.columns if 'OutTime' in col]].head())


### 3.2 Seleção de Features

Removemos variáveis que não agregam valor ao modelo:
1. **Set Points (.SP)**: Já capturamos sua informação através do erro.
2. **Variáveis Constantes**: Variância zero.
3. **Variáveis Redundantes**: Alta multicolinearidade (>0.95), mantendo apenas uma do par.



In [ ]:
# Remover Set Points
df_clean = df_eng.drop(columns=sp_vars)

# Remover Variáveis Constantes
const_cols = [c for c in df_clean.columns if df_clean[c].std() == 0]
df_clean = df_clean.drop(columns=const_cols)
print(f"Removidas {len(const_cols)} colunas constantes: {const_cols}")

# Remover Multicolinearidade (Threshold 0.95)
# Calcular matriz de correlação absoluta
corr_abs = df_clean.select_dtypes(include=[np.number]).corr().abs()

# Selecionar triângulo superior
upper = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))

# Identificar colunas com correlação > 0.95
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]

# Não remover o target!
if target in to_drop:
    to_drop.remove(target)

df_clean = df_clean.drop(columns=to_drop)
print(f"Removidas {len(to_drop)} colunas por alta colinearidade (>0.95): {to_drop}")

print(f"Shape final após limpeza: {df_clean.shape}")


## 4. Preparação Final para Machine Learning

Nesta etapa, garantimos que o dataset está pronto para ser consumido pelos algoritmos do scikit-learn.



In [ ]:
# Tratamento de Nulos
# Como é série temporal, usamos preenchimento para frente (Forward Fill) 
# para propagar o último estado válido, simulando a realidade de leitura de sensores.
if df_clean.isnull().sum().sum() > 0:
    print(f"Encontrados nulos. Preenchendo com ffill...")
    df_clean = df_clean.ffill().bfill() # bfill para garantir início
else:
    print("Dataset sem valores nulos.")

# Separação Features (X) e Target (y)
# Remover Timestamp pois já extraímos features temporais e não podemos passar datetime cru para maioria dos modelos
X = df_clean.drop(columns=['Timestamp', target])
y = df_clean[target]

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")

# Verificação final de tipos
print("Tipos de dados em X:")
print(X.dtypes.value_counts())


## 5. Caminhos de Machine Learning (Proposta)

Não realizaremos o treinamento neste notebook, mas apresentamos três abordagens viáveis e tecnicamente justificadas para modelar este problema.

### Caminho 1: Regressão Linear Regularizada (Lasso/ElasticNet)
*   **Adequação**: Serve como um *baseline* robusto. A regularização (L1/L2) lida bem com a multicolinearidade residual e seleciona features automaticamente.
*   **Prós**: Alta interpretabilidade (coeficientes diretos), rápido treinamento.
*   **Contras**: Assume relações lineares, o que pode ser insuficiente para a complexidade química de um reator.
*   **Sensibilidade**: Sensível a outliers (menos se usar Huber Loss) e exige escalonamento (StandardScaler) prévio.
*   **Implementação**: `sklearn.linear_model.ElasticNetCV`.

### Caminho 2: Modelos Baseados em Árvores (XGBoost / Random Forest)
*   **Adequação**: Ideal para dados tabulares industriais. Captura não-linearidades e interações complexas entre variáveis (ex: pressão x temperatura) sem necessidade de engenharia de features explícita para interações.
*   **Prós**: Robusto a outliers, não exige normalização, lida bem com dados mistos. Geralmente apresenta melhor performance (SOTA para tabulares).
*   **Contras**: Menor interpretabilidade direta que modelos lineares (mas explicável via SHAP values).
*   **Sensibilidade**: Baixa sensibilidade a multicolinearidade e ruído.
*   **Implementação**: `xgboost.XGBRegressor` ou `sklearn.ensemble.RandomForestRegressor`.

### Caminho 3: Métodos de Ensemble Robustos (Voting Regressor)
*   **Adequação**: Combina a estabilidade de modelos lineares com a capacidade não-linear de árvores ou SVR.
*   **Prós**: Reduz a variância do erro e tende a generalizar melhor que modelos individuais.
*   **Contras**: Maior custo computacional e complexidade de manutenção.
*   **Sensibilidade**: Muito robusto, pois erros de um modelo podem ser compensados por outro.
*   **Implementação**: `sklearn.ensemble.VotingRegressor` combinando (Linear, XGBoost, SVR).

